# YOLOV12 — Benchmark on OOD Dataset (22 classes)


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov12n", "yolov12s", "yolov12m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov12.csv"
PERCLASS_CSV = "benchmark_yolov12_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov12n.pt",
    "yolov12s.pt",
    "yolov12m.pt",
]

In [ ]:
def benchmark_model(model_name):
    """Train a model on OOD and return benchmark metrics."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {model_name}")
    print(f"{'='*60}")

    model = YOLO(model_name)

    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        name=f"{model_name.replace('.pt', '')}_ood",
        patience=PATIENCE,
        save=True,
        plots=True,
    )

    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))

    metrics = best_model.val(data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE)

    test_img_dir = TEST_IMG_DIR
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    size_mb = best_path.stat().st_size / 1e6 if best_path.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }

    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, ap in enumerate(ap50_per_class):
        per_class[CLASS_NAMES[i]] = round(float(ap), 4)

    del model, best_model
    gc.collect()
    torch.cuda.empty_cache()

    return row, per_class

In [ ]:
rows = []
all_per_class = {}

for model_name in MODELS:
    try:
        row, per_class = benchmark_model(model_name)
        rows.append(row)
        all_per_class[row["model"]] = per_class

        print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
        print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
        print(f"  Precision     : {row['precision']}")
        print(f"  Recall        : {row['recall']}")
        print(f"  F1            : {row['F1']}")
        print(f"  Speed         : {row['speed_ms/img']} ms/img")
        print(f"  Size          : {row['size_MB']} MB")
        print(f"  Params        : {row['params_M']} M")
    except Exception as e:
        print(f"  SKIPPED {model_name}: {e}")
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False)

df_pc = pd.DataFrame(all_per_class).T
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV}")
print(f"Saved -> {PERCLASS_CSV}")

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)

print("=" * 60)
print("  YOLOv12 BENCHMARK — Indoor Dataset (22 classes)")
print("=" * 60)

styled = (
    df.style
    .set_caption("YOLOv12 Benchmark — Indoor Dataset")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "precision":    "{:.4f}",
        "recall":       "{:.4f}",
        "F1":           "{:.4f}",
        "speed_ms/img": "{:.1f} ms",
        "size_MB":      "{:.1f} MB",
        "params_M":     "{:.1f} M",
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"], color="#2d6a2e")
    .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="#1a5276")
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("padding", "8px")]},
    ])
    .hide(axis="index")
)
display(styled)

In [ ]:
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)

print("Per-class mAP@0.5 across YOLOv12 variants")
print("-" * 60)

styled_pc = (
    df_pc.style
    .set_caption("Per-Class mAP@0.5 — YOLOv12 Benchmark")
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .set_properties(**{"text-align": "center", "font-size": "12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
    ])
)
display(styled_pc)

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
axes[0].set_xlabel("mAP@0.5")
axes[0].set_title("Accuracy (mAP@0.5)")
axes[0].set_xlim(0, 1)

axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
axes[1].set_xlabel("ms / image")
axes[1].set_title("Inference Speed")

axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
axes[2].set_xlabel("MB")
axes[2].set_title("Model Size")

plt.suptitle("YOLOv12 Benchmark — Indoor Dataset", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_yolov12_chart.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov12n", "yolov12s", "yolov12m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov12.csv"
PERCLASS_CSV = "benchmark_yolov12_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov12n.pt",
    "yolov12s.pt",
    "yolov12m.pt",
]